<a href="https://www.kaggle.com/code/lamontesmith/llmops-loop-evaluating-operationalizing-ai-agent?scriptVersionId=309452494" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Week 6: Evaluating & Operationalizing AI Agents

**Author:** Lamonte Smith  
**Program:** Interview Kickstart Applied Agentic AI  
**GitHub:** [github.com/LSmithPMP/ik-agentic-ai-assignments](https://github.com/LSmithPMP/ik-agentic-ai-assignments)  
**Date:** April 2026

---

## Overview

This notebook practices the full **LLMOps loop** on a working signup email agent:

- **Inspect** — review baseline agent behavior
- **Improve** — rewrite the prompt with explicit guardrails
- **Expand** — add 7 edge case examples to the golden dataset
- **Evaluate** — score with 3 metrics including a custom LLM-as-a-Judge

### The Agent
Takes a user signup record and generates:
1. An ICP (Ideal Customer Profile) classification: `high | medium | low | unknown`
2. A personalized welcome email

### Tools
- **LangChain** — agent orchestration
- **LangSmith** — tracing and online evaluation
- **DeepEval** — offline evaluation metrics
- **OpenAI GPT-4o-mini** — LLM backbone

---

## Key Results

| Metric | Score |
|--------|-------|
| ICP Fit Exact Match | 62.5% (5/8 cases) |
| Email Quality Avg | 0.81 |
| Personalization Pass Rate | 75% (strict 0/1) |

> **Note:** ICP match dropped from 83% (6-case dataset) to 62% (8-case dataset) because the expanded dataset tests harder edge cases — not a regression in prompt quality.

## Step 1: Install Dependencies

In [ ]:
!pip install -q langchain langchain_openai langchain_core langsmith deepeval pandas

## Step 2: Configure API Keys

**Kaggle setup:**
1. Go to **Add-ons → Secrets** in the Kaggle notebook toolbar
2. Add `OPENAI_API_KEY` with your key
3. Add `LANGSMITH_API_KEY` with your LangSmith key
4. Toggle access on for both

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["OPENAI_API_KEY"] = secrets.get_secret("OPENAI_API_KEY")
    os.environ["LANGSMITH_API_KEY"] = secrets.get_secret("LANGSMITH_API_KEY")
    print("API keys loaded from Kaggle Secrets")
except Exception:
    os.environ["OPENAI_API_KEY"] = input("Enter OpenAI API key: ")
    os.environ["LANGSMITH_API_KEY"] = input("Enter LangSmith API key: ")
    print("API keys set manually")

os.environ["LANGSMITH_TRACING"] = "true"
print("LangSmith tracing enabled")

## Step 3: Create the Agent with Improved Prompt

In [ ]:
from langchain.agents import create_agent
from pydantic import BaseModel, Field
from typing import Literal


class SignupEmailOutput(BaseModel):
    to: str = Field(description="Email address")
    icp_fit: Literal["high", "medium", "low", "unknown"] = Field(description="ICP fit classification")
    reason: str = Field(description="Brief reasoning for classification")
    subject: str = Field(description="Email subject line")
    body: str = Field(description="Email body text")


SYSTEM_PROMPT = """
You are an onboarding specialist for Glop, a developer productivity SaaS tool.

Your job is to process a new user signup and return a structured response containing:
1. An ICP (Ideal Customer Profile) classification
2. A short, personalized welcome email

ICP CLASSIFICATION RULES
Classify icp_fit as one of: high | medium | low | unknown

high    -> Engineering leader (VP, Director, CTO, Staff/Principal Engineer)
           at a tech or SaaS company with 50+ employees.
medium  -> Developer, individual contributor, or non-engineering leader at a
           tech company. Or a clearly technical role at a smaller company.
low     -> Student, freelancer, spam-looking email, or role completely outside
           engineering and technology.
unknown -> Insufficient signal: missing role, company, AND industry.

Always tie the reason to specific available fields.

EMAIL WRITING RULES
- Use first_name if available
- If first_name is missing but role/company ARE present, reference those instead
- NEVER default to 'Hi there' or 'Dear User' when other fields exist
- Never invent facts not in the signup record
- Professional but human tone; target 80-120 words
- Single clear CTA; sign off from 'The Glop Team'
- Spam/nonsensical input: polite but minimal response
"""

agent = create_agent(
    "openai:gpt-4o-mini",
    tools=[],
    system_prompt=SYSTEM_PROMPT,
    response_format=SignupEmailOutput,
)

print("Agent created with improved prompt")

## Step 4: Test the Agent — Baseline Case

In [ ]:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": """Process this signup:
email: sarah@acmecorp.com
first_name: Sarah
company_name: Acme Corp
role: VP of Engineering
industry: SaaS
company_size: 500"""
    }]
})

output = result["structured_response"]
print(f"ICP Fit: {output.icp_fit}")
print(f"Reason:  {output.reason}")
print(f"Subject: {output.subject}")
print(f"Body:\n{output.body}")

## Step 5: Golden Dataset — 8 Cases

Seven edge cases added to the original baseline covering all required patterns:

| Case | Edge Pattern |
|------|--------------|
| Sarah — VP Eng, Acme Corp | Baseline happy path |
| J. Chen — Sr. SWE, Stripe (no name) | Missing first name with rich fields |
| Marcus — DevOps, Accenture (gmail) | Generic email domain with full profile |
| j.doe — no fields | Incomplete context — all fields blank |
| Alex — Co-Founder, Stealth Startup | Stealth/unknown company |
| Priya — CS Student | Unclear role — student |
| Mike — Freelance Developer | Unclear role — freelancer |
| Tom — Operations Manager, Manufacturing | Non-technical role in non-tech industry |

In [ ]:
# Golden dataset — all 8 cases with expected outputs
golden_dataset = [
    {
        "input": "email: sarah@acmecorp.com\nfirst_name: Sarah\ncompany_name: Acme Corp\nrole: VP of Engineering\nindustry: SaaS\ncompany_size: 500",
        "icp_fit": "high",
        "note": "Baseline — all fields present, clear high ICP"
    },
    {
        "input": "email: j.chen@gmail.com\nfirst_name: \ncompany_name: Stripe\nrole: Senior Software Engineer\nindustry: FinTech\ncompany_size: 4000",
        "icp_fit": "medium",
        "note": "Missing name — must reference Stripe or role, NOT 'Hi there'"
    },
    {
        "input": "email: marcus.j@gmail.com\nfirst_name: Marcus\ncompany_name: Accenture\nrole: DevOps Engineer\nindustry: Consulting\ncompany_size: 500000",
        "icp_fit": "medium",
        "note": "Generic email domain offset by rich role/company context"
    },
    {
        "input": "email: j.doe@gmail.com\nfirst_name: \ncompany_name: \nrole: \nindustry: \ncompany_size: ",
        "icp_fit": "unknown",
        "note": "Zero signal — all fields blank"
    },
    {
        "input": "email: alex@stealth.io\nfirst_name: Alex\ncompany_name: Stealth Startup\nrole: Co-Founder\nindustry: Unknown\ncompany_size: 3",
        "icp_fit": "medium",
        "note": "Stealth company — no external context"
    },
    {
        "input": "email: priya.k@university.edu\nfirst_name: Priya\ncompany_name: State University\nrole: CS Student\nindustry: Education\ncompany_size: 5000",
        "icp_fit": "low",
        "note": "Student — no budget authority"
    },
    {
        "input": "email: consultant99@hotmail.com\nfirst_name: Mike\ncompany_name: Self-Employed\nrole: Freelance Developer\nindustry: Consulting\ncompany_size: 1",
        "icp_fit": "low",
        "note": "Freelancer — solo operator"
    },
    {
        "input": "email: tdavis@globalmanufacturing.com\nfirst_name: Tom\ncompany_name: Global Manufacturing Inc.\nrole: Operations Manager\nindustry: Manufacturing\ncompany_size: 2000",
        "icp_fit": "low",
        "note": "Non-technical role in non-tech industry"
    },
]

print(f"Golden dataset: {len(golden_dataset)} cases loaded")
for i, case in enumerate(golden_dataset):
    print(f"  Case {i+1}: {case['note']} | Expected: {case['icp_fit']}")

## Step 6: Create Evaluators — Including Custom Personalization Judge

In [ ]:
from deepeval.metrics import ExactMatchMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

# Metric 1: ICP Fit Exact Match
icp_fit_metric = ExactMatchMetric(threshold=1.0)
icp_fit_metric.include_reason = True

# Metric 2: Email Quality
email_metric = GEval(
    name="Email Quality",
    criteria="Evaluate if the email is professional, personalized (uses the "
             "person's name/role/company), concise (under 150 words), and does "
             "not hallucinate facts not present in the input.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model="gpt-4o-mini",
    threshold=0.5
)
email_metric.include_reason = True

# Metric 3: Personalization Quality — strict 0/1 binary
personalization_metric = GEval(
    name="Personalization Quality",
    criteria="""
    Signs of GOOD personalization (score 1):
    - Uses name, role, or company from the signup record
    - Sounds written for this specific user, not any user
    - Avoids details not present in the input
    - Avoids generic openers when name/role/company IS available

    Signs of POOR personalization (score 0):
    - Generic greetings when input fields were available
    - Hallucinated details not in the signup record
    - Same tone and structure regardless of user type
    - Misses obvious personalization opportunities

    CRITICAL: When first_name is missing but role/company ARE available,
    the email MUST reference role or company. Defaulting to 'Hi there'
    when other fields exist is a FAIL (score 0).

    Return ONLY 0 or 1. No partial scores.
    """,
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model="gpt-4o-mini",
    threshold=0.5,
    strict_mode=True
)
personalization_metric.include_reason = True

print("All 3 evaluators ready")
print("  - ICP Fit: ExactMatch (threshold=1.0)")
print("  - Email Quality: GEval (threshold=0.5)")
print("  - Personalization Quality: GEval strict 0/1")

## Step 7: Run Full Offline Evaluation Loop

In [ ]:
import json

icp_fit_scores = []
email_scores = []
personalization_scores = []

print(f"Running eval on {len(golden_dataset)} cases...\n")

for case in golden_dataset:
    input_msg = {"messages": [{"role": "user", "content": f"Process this signup:\n{case['input']}"}]}
    result = agent.invoke(input_msg)
    actual = result["structured_response"]

    # ICP Fit Exact Match
    icp_tc = LLMTestCase(
        input=json.dumps(input_msg),
        actual_output=actual.icp_fit,
        expected_output=case["icp_fit"]
    )
    icp_fit_metric.measure(icp_tc)
    icp_fit_scores.append(icp_fit_metric.score)

    # Email Quality
    email_tc = LLMTestCase(
        input=json.dumps(input_msg),
        actual_output=actual.body
    )
    email_metric.measure(email_tc)
    email_scores.append(email_metric.score)

    # Personalization Quality
    pers_tc = LLMTestCase(
        input=json.dumps(input_msg),
        actual_output=actual.body
    )
    personalization_metric.measure(pers_tc)
    personalization_scores.append(personalization_metric.score)

    print(f"ICP: {actual.icp_fit:8} | Email: {email_metric.score:.2f} | Personalization: {personalization_metric.score} | {actual.subject}")
    print(f"  {personalization_metric.reason}\n")

print("=" * 70)
print(f"ICP Fit Exact Match Rate:    {sum(icp_fit_scores)/len(icp_fit_scores):.2%}")
print(f"Email Quality Avg Score:     {sum(email_scores)/len(email_scores):.2f}")
print(f"Personalization Pass Rate:   {sum(personalization_scores)/len(personalization_scores):.0%}")
print("=" * 70)

## Results Summary

| Metric | Score |
|--------|-------|
| ICP Fit Exact Match Rate | 62.50% (5/8) |
| Email Quality Avg Score | 0.81 |
| Personalization Pass Rate | 75% (6/8) |

### Key Findings
- **Sarah (VP Eng):** Best case — 0.94 email quality, 1/1 personalization
- **J. Chen (missing name):** ICP miss (high vs medium) + personalization fail
- **Tom (non-tech):** ICP miss (medium vs low) — non-technical role signal underweighted  
- **j.doe (no fields):** Expected 0 personalization — zero signal edge case

### The Core LLMOps Pattern
```
Online eval catches issues → Golden dataset grows → Offline eval prevents regressions → Agent improves
```

---
*Lamonte Smith · [github.com/LSmithPMP](https://github.com/LSmithPMP) · Interview Kickstart Applied Agentic AI*